# fact_player_facets.py

## Purpose
Creates a long-format table (`facet`, `value`) from the 6 card facets in `fact_player_snapshot`. Used in Power BI for the radar chart.

## Source
- Silver: `fifa.silver.fact_player_snapshot`

## Output
- Table: `fifa.silver.fact_player_facets` (partitioned by `snapshot_date`)

## Columns
- `player_snapshot_key` (STRING)
- `player_id` (INT)
- `snapshot_date` (DATE)
- `faceta` (STRING): Pace, Shooting, Passing, Dribbling, Defending, Physicality
- `valor` (INT)

## Strategy
- First load: `overwrite`
- Subsequent loads: `MERGE` on `(player_snapshot_key, faceta)`
- Partitioned by `snapshot_date`

## Why a long-format table?
Power BI's radar chart requires one row per facet, not one column per facet. The `stack(6, ...)` expression unpivots the 6 facets into rows.

## Grain
- One row per `(player_snapshot_key, faceta)`.
- A player with 6 facets → 6 rows.

In [0]:
from pyspark.sql.functions import col, expr
from delta.tables import DeltaTable

# ===== Read fact table =====
fact = spark.read.table("fifa.silver.fact_player_snapshot")

# ===== Unpivot the 6 card facets into long format =====
facets = (
    fact
    .select(
        col("player_snapshot_key"),
        col("player_id"),
        col("snapshot_date"),
        expr("""
            stack(6,
                'Pace', pace,
                'Shooting', shooting,
                'Passing', passing,
                'Dribbling', dribbling,
                'Defending', defending,
                'Physicality', physicality
            ) AS (faceta, valor)
        """)
    )
    .filter(col("valor").isNotNull())
)

# ===== Create Silver table =====
spark.sql("""
CREATE TABLE IF NOT EXISTS fifa.silver.fact_player_facets (
    player_snapshot_key STRING,
    player_id INT,
    snapshot_date DATE,
    faceta STRING,
    valor INT
)
USING DELTA
PARTITIONED BY (snapshot_date);
""")

# ===== Load: overwrite on first load, MERGE afterwards =====
target_table = "fifa.silver.fact_player_facets"

table_exists = spark.catalog.tableExists(target_table)
row_count = 0

if table_exists:
    row_count = spark.read.table(target_table).count()

if not table_exists or row_count == 0:
    # First load: overwrite
    print("First load detected. Writing with overwrite...")
    (facets
     .write
     .mode("overwrite")
     .option("mergeSchema", "true")
     .partitionBy("snapshot_date")
     .saveAsTable(target_table)
    )
    print(f"Table created with {spark.read.table(target_table).count()} rows")

else:
    # Subsequent loads: MERGE by (player_snapshot_key, faceta)
    print(f"Existing table with {row_count} rows. Applying MERGE...")

    target = DeltaTable.forName(spark, target_table)

    (
        target.alias("t")
        .merge(
            facets.alias("s"),
            """
            t.player_snapshot_key = s.player_snapshot_key
            AND t.faceta = s.faceta
            """
        )
        .whenMatchedUpdate(set={"valor": "s.valor"})
        .whenNotMatchedInsert(values={
            "player_snapshot_key": "s.player_snapshot_key",
            "player_id": "s.player_id",
            "snapshot_date": "s.snapshot_date",
            "faceta": "s.faceta",
            "valor": "s.valor"
        })
        .execute()
    )

    print(f"MERGE completed. Total rows: {spark.read.table(target_table).count()}")

LIMIT 10 of the fact_player_facets table. Quick verification that we have entered the raw data into our table.

In [0]:
%sql
select * from fifa.silver.fact_player_facets limit 10;